# Promedio de tiempos reales y atrasos hacia Estación Corregidora

Este notebook complementa `RutasQroBus.ipynb`. Parte de los horarios programados del GTFS local, consulta **Google Maps Routes API** para obtener la ETA en transporte público desde las paradas hacia la estación Corregidora, conserva observaciones históricas y calcula una corrección promedio.

El resultado principal es `data/tiempos_corregidos_google.csv`. `RutasQroBus.ipynb` lo consume de manera opcional para que el filtro de 30 minutos use el tiempo corregido.


## 1. Alcance y definición de las métricas

Se manejan tres medidas distintas:

1. **Tiempo GTFS programado**: suma de los tiempos entre paradas según `stop_times.txt`.
2. **ETA de Google Transit**: duración estimada al momento de la consulta; puede incluir espera, caminata y transbordos.
3. **Atraso estimado (proxy)**: `max(ETA Google - tiempo GTFS, 0)`. Se promedia por ruta y se suma al cálculo GTFS, como se solicitó.

> **Limitación importante:** Google Routes API no entrega un campo de “atraso del camión” ni permite seleccionar un modelo de tráfico cuando `travelMode="TRANSIT"`. La ETA puede incorporar información actualizada cuando el proveedor de transporte la comparte, pero el diferencial contra GTFS también puede contener espera, caminata o transbordos. Por ello se etiqueta como **proxy**, no como atraso operativo observado. Para atraso vehicular exacto se necesita el feed GTFS-Realtime `TripUpdates` de la agencia.

Documentación oficial consultada:

- [Rutas en transporte público](https://developers.google.com/maps/documentation/routes/transit-route)
- [Matriz de rutas en transporte público](https://developers.google.com/maps/documentation/routes/transit-rm)
- [Límites y facturación](https://developers.google.com/maps/documentation/routes/usage-and-billing)
- [Políticas y atribución de Routes API](https://developers.google.com/maps/documentation/routes/policies)


## 2. Configuración segura

La llave nunca se escribe en el notebook. Se carga desde el archivo `.env` ubicado en la raíz del repositorio. Copia `.env.example` como `.env` y completa:

```dotenv
GOOGLE_MAPS_API_KEY=tu_llave
EJECUTAR_GOOGLE_MAPS=true
UMBRAL_ANALISIS_MIN=30
```

El cargador no sobrescribe variables que ya existan en el sistema. `.env` está excluido por `.gitignore`; solo `.env.example`, que no contiene secretos, puede versionarse. La consulta es facturable y permanece apagada salvo que `EJECUTAR_GOOGLE_MAPS=true`.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import os
import sys
import time
import warnings

import numpy as np
import pandas as pd
import requests
from IPython.display import display

# Rutas portables: funciona desde scripts/ y desde la raíz del repositorio.
DATA_DIR = Path("../data") if Path("../data").exists() else Path("data")
if not DATA_DIR.exists():
    raise FileNotFoundError("No se encontró la carpeta local data/.")

PROJECT_ROOT = DATA_DIR.parent.resolve()
ENV_PATH = PROJECT_ROOT / ".env"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from gtfs_network import (
    calcular_rutas_hacia_destino,
    compactar_rutas,
    filtrar_segmentos_mismo_lado_vias,
    gtfs_time_to_seconds,
    guardar_rutas_base,
    VIA_FERREA_CORREGIDORA_LATLON,
)

def cargar_env(env_path):
    """Carga pares CLAVE=VALOR sencillos sin imprimir secretos."""
    if not env_path.exists():
        return False

    for numero_linea, linea_original in enumerate(
        env_path.read_text(encoding="utf-8").splitlines(), start=1
    ):
        linea = linea_original.strip()
        if not linea or linea.startswith("#"):
            continue
        if linea.startswith("export "):
            linea = linea[7:].strip()
        if "=" not in linea:
            raise ValueError(f"Línea inválida en .env: {numero_linea}")

        clave, valor = linea.split("=", 1)
        clave, valor = clave.strip(), valor.strip()
        if len(valor) >= 2 and valor[0] == valor[-1] and valor[0] in {"'", '"'}:
            valor = valor[1:-1]
        os.environ.setdefault(clave, valor)

    return True

ENV_CARGADO = cargar_env(ENV_PATH)

CORREGIDORA_LAT = 20.600611
CORREGIDORA_LON = -100.402184

API_KEY = os.getenv("GOOGLE_MAPS_API_KEY", "").strip() or None
EJECUTAR_API = os.getenv("EJECUTAR_GOOGLE_MAPS", "false").strip().lower() == "true"
UMBRAL_ANALISIS_MIN = float(os.getenv("UMBRAL_ANALISIS_MIN", "30"))
PENALIZACION_TRANSBORDO_MIN = float(
    os.getenv("PENALIZACION_TRANSBORDO_MIN", "5")
)
TAMANO_LOTE = 100
ESPERA_ENTRE_LOTES_SEG = 0.25
RETENCION_DIAS = 30

# None consulta todos los orígenes alcanzables. Use un entero para una prueba controlada.
MAX_PARADAS = None

if EJECUTAR_API and not API_KEY:
    raise RuntimeError(
        "EJECUTAR_GOOGLE_MAPS=true, pero falta GOOGLE_MAPS_API_KEY."
    )
if PENALIZACION_TRANSBORDO_MIN < 0:
    raise ValueError("PENALIZACION_TRANSBORDO_MIN no puede ser negativa.")

print(f"Archivo .env cargado: {ENV_CARGADO} ({ENV_PATH})")
print(f"API key configurada: {bool(API_KEY)}")
print(f"Directorio de datos: {DATA_DIR.resolve()}")
print(f"Consulta a Google habilitada: {EJECUTAR_API}")
print(f"Penalización por transbordo: {PENALIZACION_TRANSBORDO_MIN:g} min")


Archivo .env cargado: True (/Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/.env)
API key configurada: True
Directorio de datos: /Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/data
Consulta a Google habilitada: True
Penalización por transbordo: 5 min


## 3. Lectura y validación de los archivos GTFS

Se leen los cuatro archivos desde `data/` y se validan las columnas mínimas antes de calcular. Esta validación falla temprano si cambia el esquema de la fuente.


In [2]:
trips = pd.read_csv(DATA_DIR / "trips.txt")
stop_times = pd.read_csv(DATA_DIR / "stop_times.txt")
stops = pd.read_csv(DATA_DIR / "stops.txt")
routes = pd.read_csv(DATA_DIR / "routes.txt")

columnas_requeridas = {
    "trips": {"trip_id", "route_id"},
    "stop_times": {
        "trip_id", "stop_id", "stop_sequence", "arrival_time", "departure_time"
    },
    "stops": {"stop_id", "stop_name", "stop_lat", "stop_lon"},
    "routes": {"route_id", "route_short_name", "route_long_name"},
}
tablas = {
    "trips": trips,
    "stop_times": stop_times,
    "stops": stops,
    "routes": routes,
}

for nombre, requeridas in columnas_requeridas.items():
    faltantes = requeridas - set(tablas[nombre].columns)
    if faltantes:
        raise ValueError(f"{nombre} no contiene las columnas: {sorted(faltantes)}")

if stops["stop_id"].duplicated().any():
    warnings.warn("Hay stop_id duplicados; se conservará el primer registro.")
    stops = stops.drop_duplicates("stop_id", keep="first").copy()

print(
    f"{len(stops):,} paradas, {len(routes):,} rutas, "
    f"{len(trips):,} viajes y {len(stop_times):,} horarios cargados."
)


2,694 paradas, 217 rutas, 32,536 viajes y 1,326,009 horarios cargados.


## 4. Tiempo programado hacia Corregidora

Los horarios GTFS admiten horas mayores a 24:00. Primero se convierten a segundos sin perder el cambio de día. Después se construye un grafo dirigido de segmentos consecutivos y se aplica Dijkstra sobre estados `(parada, ruta)` para encontrar el menor tiempo programado desde cada parada hacia Corregidora, incluyendo la penalización configurada por transbordo. El resultado completo se guarda en `data/rutas_base_gtfs.csv`; `RutasQroBus.ipynb` lo reutiliza sin ejecutar Dijkstra otra vez.

Para evitar que una observación extrema gobierne un segmento, el peso de cada combinación `origen-destino-ruta` es su **mediana** programada.


In [3]:
stop_times_work = stop_times.copy()
stop_times_work["arrival_sec"] = gtfs_time_to_seconds(
    stop_times_work["arrival_time"]
)
stop_times_work["departure_sec"] = gtfs_time_to_seconds(
    stop_times_work["departure_time"]
)
stop_times_work = stop_times_work.merge(
    trips[["trip_id", "route_id"]], on="trip_id", how="left", validate="many_to_one"
)
stop_times_work = stop_times_work.sort_values(
    ["trip_id", "stop_sequence"], kind="stable"
)

stop_times_work["next_stop_id"] = (
    stop_times_work.groupby("trip_id", sort=False)["stop_id"].shift(-1)
)
stop_times_work["next_arrival_sec"] = (
    stop_times_work.groupby("trip_id", sort=False)["arrival_sec"].shift(-1)
)

segmentos = stop_times_work.dropna(
    subset=["next_stop_id", "next_arrival_sec", "route_id"]
).copy()

segmentos["next_stop_id"] = segmentos["next_stop_id"].astype(stops["stop_id"].dtype)

segmentos["tiempo_segmento_min"] = (
    segmentos["next_arrival_sec"] - segmentos["departure_sec"]
) / 60

segmentos = segmentos[
    segmentos["tiempo_segmento_min"].between(0, 180, inclusive="both")
].copy()

segmentos_medianos = (
    segmentos.groupby(
        ["stop_id", "next_stop_id", "route_id"], as_index=False
    )["tiempo_segmento_min"]
    .median()
)

coords = stops[["stop_lat", "stop_lon"]].to_numpy()
distancia_cuadrada = (
    (coords[:, 0] - CORREGIDORA_LAT) ** 2
    + (coords[:, 1] - CORREGIDORA_LON) ** 2
)
corregidora_idx = int(np.argmin(distancia_cuadrada))
corregidora_stop_id = stops.iloc[corregidora_idx]["stop_id"]
corregidora_stop_name = stops.iloc[corregidora_idx]["stop_name"]

print(
    "Parada GTFS más cercana a Corregidora:",
    corregidora_stop_id,
    "-",
    corregidora_stop_name,
)


Parada GTFS más cercana a Corregidora: 3025 - Estío/Calle Dr. Manuel Domínguez


In [4]:
resultados_base = calcular_rutas_hacia_destino(
    segmentos=segmentos_medianos,
    stops=stops,
    routes=routes,
    destino_stop_id=corregidora_stop_id,
    penalizacion_transbordo_min=PENALIZACION_TRANSBORDO_MIN,
)

RUTAS_BASE_PATH = DATA_DIR / "rutas_base_gtfs.csv"
guardar_rutas_base(resultados_base, RUTAS_BASE_PATH)

# Formato compatible con el cálculo histórico de las correcciones de Google.
df_base = resultados_base.rename(
    columns={"tiempo_red_min": "tiempo_gtfs_min"}
).copy()
df_base["camino_stop_ids"] = df_base["camino_stop_ids"].apply(
    lambda valores: "|".join(map(str, valores))
)
df_base["rutas_camino"] = df_base["rutas_por_segmento"].apply(
    lambda valores: "|".join(map(str, compactar_rutas(valores)))
)
route_name_map = routes.set_index("route_id")["route_short_name"].to_dict()

print(f"{len(df_base):,} paradas tienen camino programado hacia Corregidora.")
print(f"Rutas base exportadas para RutasQroBus: {RUTAS_BASE_PATH}")
display(df_base.head(10))


2,259 paradas tienen camino programado hacia Corregidora.
Rutas base exportadas para RutasQroBus: ../data/rutas_base_gtfs.csv


,stop_id,tiempo_gtfs_min,route_id_principal,route_short_name,num_transbordos,num_paradas_camino,num_rutas_camino,tipo_conexion,itinerario_route_ids,itinerario_rutas,paradas_transbordo,camino_stop_ids,rutas_por_segmento,penalizacion_transbordo_config_min,stop_name,stop_lat,stop_lon,rutas_camino
0,3025,0.000000,NaN,None,0,1,0,Destino,,,,3025,[],5.0,Estío/Calle Dr. Manuel Domínguez,20.599573,-100.400492,
1,2341,1.533333,344.0,C54,0,2,1,Directa,344,C54,,2341|3025,[344],5.0,Av. Felipe Ángeles/Av. San Roque,20.603829,-100.399494,344
2,3295,2.216667,344.0,C54,0,3,1,Directa,344,C54,,3295|2341|3025,"[344, 344]",5.0,Felipe Ángeles/Fraternidad,20.605753,-100.399474,344
3,2340,2.733333,344.0,C54,0,4,1,Directa,344,C54,,2340|3295|2341|3025,"[344, 344, 344]",5.0,Av. Felipe Ángeles/Plan de Ayala Poniente,20.607204,-100.399456,344
4,3294,3.583333,344.0,C54,0,5,1,Directa,344,C54,,3294|2340|3295|2341|3025,"[344, 344, 344, 344]",5.0,Av. Felipe Ángeles/Calle del Porvenir,20.609586,-100.399427,344
5,2339,4.200000,344.0,C54,0,6,1,Directa,344,C54,,2339|3294|2340|3295|2341|3025,"[344, 344, 344, 344, 344]",5.0,Av. Felipe Ángeles/Felipe Ángeles 225,20.611300,-100.399656,344
6,2338,4.700000,344.0,C54,0,7,1,Directa,344,C54,,2338|2339|3294|2340|3295|2341|3025,"[344, 344, 344, 344, 344, 344]",5.0,Av. Felipe Ángeles/Estadística,20.612684,-100.399821,344
7,4256,5.366667,344.0,C54,0,8,1,Directa,344,C54,,4256|2338|2339|3294|2340|3295|2341|3025,"[344, 344, 344, 344, 344, 344, 344]",5.0,Epigmenio González/Departamental Parques,20.612535,-100.401796,344
8,3363,8.033333,344.0,C54,0,9,1,Directa,344,C54,,3363|4256|2338|2339|3294|2340|3295|2341|3025,"[344, 344, 344, 344, 344, 344, 344, 344]",5.0,Prol. Tecnológico/Calle San Joaquín,20.612622,-100.409849,344
9,3362,9.200000,344.0,C54,0,10,1,Directa,344,C54,,3362|3363|4256|2338|2339|3294|2340|3295|2341|3025,"[344, 344, 344, 344, 344, 344, 344, 344, 344]",5.0,Prol. Tecnológico/Ford Citelis Querétaro,20.615897,-100.410138,344


## 5. Escenario al sur de las vías, sin cruces ferroviarios

Este segundo escenario usa `2312 — Av. Tecnológico/Calle Mariano Escobedo` únicamente como referencia para seleccionar el lado sur de la vía. Antes de calcular los caminos se eliminan todos los segmentos que tocan o cruzan la línea ferroviaria. Las paradas del mismo lado situadas a diez minutos caminando o menos de la estación se conectan a un destino peatonal virtual; después se ejecuta Dijkstra sobre ese grafo reducido. Esto permite combinar autobús y último tramo a pie sin exigir una ruta que cruce las vías.

La polilínea simplificada corresponde a la `Línea Juárez` de OpenStreetMap en el entorno cubierto por los caminos de 30 minutos. Es una barrera analítica, no un levantamiento de ingeniería. Datos cartográficos © [colaboradores de OpenStreetMap](https://www.openstreetmap.org/copyright), disponibles bajo ODbL.


In [5]:
REFERENCIA_LADO_SUR_STOP_ID = "2312"
DESTINO_PEATONAL_SUR_ID = "ESTACION_CORREGIDORA_SUR"
RUTA_CAMINATA_ID = "CAMINATA"
MARGEN_SEGURIDAD_VIA_M = 15.0
VELOCIDAD_CAMINATA_M_MIN = 80.0
MAX_CAMINATA_FINAL_MIN = 10.0

segmentos_bus_sur = filtrar_segmentos_mismo_lado_vias(
    segmentos=segmentos_medianos,
    stops=stops,
    via_latlon=VIA_FERREA_CORREGIDORA_LATLON,
    referencia_stop_id=REFERENCIA_LADO_SUR_STOP_ID,
    margen_m=MARGEN_SEGURIDAD_VIA_M,
)

via = np.asarray(VIA_FERREA_CORREGIDORA_LATLON, dtype=float)
orden_via = np.argsort(via[:, 1])
via_lat, via_lon = via[orden_via, 0], via[orden_via, 1]
referencia_sur = stops[
    stops["stop_id"].astype(str).eq(REFERENCIA_LADO_SUR_STOP_ID)
].iloc[0]
signo_sur = np.sign(
    referencia_sur["stop_lat"]
    - np.interp(referencia_sur["stop_lon"], via_lon, via_lat)
)
stops_sur = stops.copy()
stops_sur["distancia_firmada_via_lat"] = (
    stops_sur["stop_lat"]
    - np.interp(stops_sur["stop_lon"], via_lon, via_lat)
) * signo_sur
stops_sur = stops_sur[
    stops_sur["distancia_firmada_via_lat"]
    >= MARGEN_SEGURIDAD_VIA_M / 111_320.0
].copy()

phi_stops = np.radians(stops_sur["stop_lat"].to_numpy())
phi_estacion = np.radians(CORREGIDORA_LAT)
delta_phi = np.radians(CORREGIDORA_LAT - stops_sur["stop_lat"].to_numpy())
delta_lon = np.radians(CORREGIDORA_LON - stops_sur["stop_lon"].to_numpy())
a = (
    np.sin(delta_phi / 2) ** 2
    + np.cos(phi_stops) * np.cos(phi_estacion) * np.sin(delta_lon / 2) ** 2
)
stops_sur["distancia_estacion_m"] = (
    6371.0088 * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a)) * 1000
)
accesos_peatonales = stops_sur[
    stops_sur["distancia_estacion_m"]
    <= MAX_CAMINATA_FINAL_MIN * VELOCIDAD_CAMINATA_M_MIN
].copy()
segmentos_caminata_candidatos = pd.DataFrame(
    {
        "stop_id": accesos_peatonales["stop_id"],
        "next_stop_id": DESTINO_PEATONAL_SUR_ID,
        "route_id": RUTA_CAMINATA_ID,
        "tiempo_segmento_min": (
            accesos_peatonales["distancia_estacion_m"]
            / VELOCIDAD_CAMINATA_M_MIN
        ),
    }
)
stops_sur_ext = pd.concat(
    [
        stops,
        pd.DataFrame(
            [
                {
                    "stop_id": DESTINO_PEATONAL_SUR_ID,
                    "stop_name": "Estación Corregidora (acceso sur)",
                    "stop_lat": CORREGIDORA_LAT,
                    "stop_lon": CORREGIDORA_LON,
                }
            ]
        ),
    ],
    ignore_index=True,
)
segmentos_caminata = filtrar_segmentos_mismo_lado_vias(
    segmentos=segmentos_caminata_candidatos,
    stops=stops_sur_ext,
    via_latlon=VIA_FERREA_CORREGIDORA_LATLON,
    referencia_stop_id=REFERENCIA_LADO_SUR_STOP_ID,
    margen_m=MARGEN_SEGURIDAD_VIA_M,
)
segmentos_sur_vias = pd.concat(
    [segmentos_bus_sur, segmentos_caminata], ignore_index=True
)
routes_sur_ext = pd.concat(
    [
        routes,
        pd.DataFrame(
            [
                {
                    "route_id": RUTA_CAMINATA_ID,
                    "route_short_name": "Caminata",
                    "route_long_name": "Último tramo a pie",
                }
            ]
        ),
    ],
    ignore_index=True,
)

resultados_sur_vias = calcular_rutas_hacia_destino(
    segmentos=segmentos_sur_vias,
    stops=stops_sur_ext,
    routes=routes_sur_ext,
    destino_stop_id=DESTINO_PEATONAL_SUR_ID,
    penalizacion_transbordo_min=PENALIZACION_TRANSBORDO_MIN,
    route_ids_sin_penalizacion={RUTA_CAMINATA_ID},
    route_ids_auxiliares={RUTA_CAMINATA_ID},
)

RUTAS_SUR_VIAS_PATH = DATA_DIR / "rutas_base_sur_vias.csv"
guardar_rutas_base(resultados_sur_vias, RUTAS_SUR_VIAS_PATH)
if REFERENCIA_LADO_SUR_STOP_ID not in set(
    resultados_sur_vias["stop_id"].astype(str)
):
    raise RuntimeError("La parada de referencia no puede llegar sin cruzar la vía.")

print(f"Segmentos generales: {len(segmentos_medianos):,}")
print(f"Segmentos de autobús conservados al sur: {len(segmentos_bus_sur):,}")
print(f"Accesos peatonales candidatos: {len(accesos_peatonales):,}")
print(f"Accesos peatonales sin cruce: {len(segmentos_caminata):,}")
print(f"Paradas alcanzables sin cruzar las vías: {len(resultados_sur_vias):,}")
print(f"Escenario alternativo exportado: {RUTAS_SUR_VIAS_PATH}")


Segmentos generales: 7,751
Segmentos de autobús conservados al sur: 3,535
Accesos peatonales candidatos: 20
Accesos peatonales sin cruce: 20
Paradas alcanzables sin cruzar las vías: 910
Escenario alternativo exportado: ../data/rutas_base_sur_vias.csv


## 6. Consulta de Google Maps Routes API

Se utiliza `computeRouteMatrix` con una sola estación destino y hasta 100 paradas origen por lote. Antes de consultar se conservan solamente las paradas cuyo tiempo GTFS es menor o igual a `UMBRAL_ANALISIS_MIN` (30 minutos por defecto). Como el atraso aplicado nunca es negativo, una parada que ya supera ese umbral no puede entrar posteriormente en el resultado de 30 minutos.

El modo `TRANSIT` se restringe preferentemente a autobús. La respuesta solo solicita los campos necesarios para controlar costo, tamaño y trazabilidad.

Cada ejecución genera una instantánea con sello UTC. Las observaciones se acumulan temporalmente en `data/google_maps_transit_observaciones.csv`; así el promedio mejora al ejecutar el notebook en distintos días y horarios. El notebook elimina automáticamente filas mayores a `RETENCION_DIAS`. Esta retención es una salvaguarda técnica, no sustituye la revisión del acuerdo vigente de Google Maps aplicable al proyecto.

La función incluye reintentos exponenciales para errores temporales (`429` y `5xx`). Los errores permanentes se conservan en la salida en vez de ocultarse.


In [6]:
ROUTE_MATRIX_URL = (
    "https://routes.googleapis.com/distanceMatrix/v2:computeRouteMatrix"
)
FIELD_MASK = (
    "originIndex,destinationIndex,duration,distanceMeters,status,condition"
)


def duracion_google_a_minutos(valor):
    if valor is None or not str(valor).endswith("s"):
        return np.nan
    return float(str(valor)[:-1]) / 60


def crear_waypoint(latitud, longitud):
    return {
        "waypoint": {
            "location": {
                "latLng": {
                    "latitude": float(latitud),
                    "longitude": float(longitud),
                }
            }
        }
    }


def consultar_lote_google(df_lote, departure_time, max_intentos=4):
    payload = {
        "origins": [
            crear_waypoint(f.stop_lat, f.stop_lon)
            for f in df_lote.itertuples(index=False)
        ],
        "destinations": [
            crear_waypoint(CORREGIDORA_LAT, CORREGIDORA_LON)
        ],
        "travelMode": "TRANSIT",
        "departureTime": departure_time,
        "transitPreferences": {"allowedTravelModes": ["BUS"]},
    }
    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": API_KEY,
        "X-Goog-FieldMask": FIELD_MASK,
    }

    for intento in range(max_intentos):
        respuesta = requests.post(
            ROUTE_MATRIX_URL,
            headers=headers,
            json=payload,
            timeout=(10, 90),
        )
        if respuesta.status_code not in {429, 500, 502, 503, 504}:
            break
        if intento == max_intentos - 1:
            break
        time.sleep(2 ** intento)

    if not respuesta.ok:
        detalle = respuesta.text[:1_000]
        raise RuntimeError(
            f"Google Routes API respondió {respuesta.status_code}: {detalle}"
        )

    try:
        elementos = respuesta.json()
    except requests.JSONDecodeError as exc:
        raise RuntimeError("La respuesta de Google no es JSON válido.") from exc

    if not isinstance(elementos, list):
        raise RuntimeError(f"Formato inesperado de Google: {elementos}")

    ahora_utc = datetime.now(timezone.utc).isoformat()
    resultados = []

    for elemento in elementos:
        indice = elemento.get("originIndex")
        if indice is None or indice >= len(df_lote):
            continue
        origen = df_lote.iloc[int(indice)]
        status = elemento.get("status") or {}
        resultados.append(
            {
                "observed_at_utc": ahora_utc,
                "departure_time_utc": departure_time,
                "stop_id": origen["stop_id"],
                "route_id_principal": origen["route_id_principal"],
                "tiempo_gtfs_min": origen["tiempo_gtfs_min"],
                "google_eta_min": duracion_google_a_minutos(
                    elemento.get("duration")
                ),
                "google_distance_m": elemento.get("distanceMeters"),
                "condition": elemento.get("condition"),
                "status_code": status.get("code", 0),
                "status_message": status.get("message"),
            }
        )

    return pd.DataFrame(resultados)


In [7]:
df_consulta = df_base[
    df_base["stop_id"].ne(corregidora_stop_id)
    & df_base["tiempo_gtfs_min"].le(UMBRAL_ANALISIS_MIN)
].copy()
if MAX_PARADAS is not None:
    df_consulta = df_consulta.head(int(MAX_PARADAS)).copy()

print(
    f"Filtro GTFS aplicado: <= {UMBRAL_ANALISIS_MIN:g} minutos. "
    f"Paradas preparadas: {len(df_consulta):,}. "
    f"Elementos facturables estimados para esta ejecución: {len(df_consulta):,}."
)

observaciones_nuevas = []
departure_time = datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")

if EJECUTAR_API:
    total_lotes = int(np.ceil(len(df_consulta) / TAMANO_LOTE))
    for numero, inicio in enumerate(
        range(0, len(df_consulta), TAMANO_LOTE), start=1
    ):
        lote = df_consulta.iloc[inicio : inicio + TAMANO_LOTE].reset_index(
            drop=True
        )
        resultado_lote = consultar_lote_google(lote, departure_time)
        observaciones_nuevas.append(resultado_lote)
        print(
            f"Lote {numero}/{total_lotes}: "
            f"{len(resultado_lote)} respuestas recibidas."
        )
        if numero < total_lotes:
            time.sleep(ESPERA_ENTRE_LOTES_SEG)
else:
    print(
        "Consulta omitida. Configure GOOGLE_MAPS_API_KEY y "
        "EJECUTAR_GOOGLE_MAPS=true para obtener una instantánea."
    )

df_nuevas = (
    pd.concat(observaciones_nuevas, ignore_index=True)
    if observaciones_nuevas
    else pd.DataFrame()
)


Candidatas de las redes general o sin cruces: Paradas preparadas: 397. Elementos facturables estimados para esta ejecución: 397.


Lote 1/4: 100 respuestas recibidas.


Lote 2/4: 100 respuestas recibidas.


Lote 3/4: 100 respuestas recibidas.


Lote 4/4: 97 respuestas recibidas.


## 7. Persistencia y control de calidad

Solo se consideran válidos los elementos con `condition="ROUTE_EXISTS"`, sin código de error y con duración numérica. El histórico conserva también los fallos para poder auditar cobertura y disponibilidad.


In [8]:
OBSERVACIONES_PATH = DATA_DIR / "google_maps_transit_observaciones.csv"

if OBSERVACIONES_PATH.exists():
    historico_previo = pd.read_csv(OBSERVACIONES_PATH)
    if "observed_at_utc" in historico_previo.columns:
        fechas = pd.to_datetime(
            historico_previo["observed_at_utc"], errors="coerce", utc=True
        )
        limite = pd.Timestamp.now(tz="UTC") - pd.Timedelta(
            RETENCION_DIAS, unit="D"
        )
        historico_previo = historico_previo[fechas.ge(limite)].copy()
else:
    historico_previo = pd.DataFrame()

df_observaciones = (
    pd.concat([historico_previo, df_nuevas], ignore_index=True)
    if not df_nuevas.empty
    else historico_previo.copy()
)

if not df_observaciones.empty:
    df_observaciones = df_observaciones.drop_duplicates(
        subset=["observed_at_utc", "stop_id"], keep="last"
    )

    # Recalcula la comparación histórica contra la red GTFS vigente.
    # Google ETA se conserva; solo se actualizan tiempo y ruta base.
    base_actual = df_base[
        ["stop_id", "tiempo_gtfs_min", "route_id_principal"]
    ]
    df_observaciones = (
        df_observaciones.drop(
            columns=["tiempo_gtfs_min", "route_id_principal"],
            errors="ignore",
        )
        .merge(base_actual, on="stop_id", how="left", validate="many_to_one")
    )
    df_observaciones.to_csv(OBSERVACIONES_PATH, index=False)
    print(f"Histórico alineado con la red GTFS actual: {OBSERVACIONES_PATH}")

if df_observaciones.empty:
    df_validas = pd.DataFrame()
    print("Todavía no existen observaciones de Google para promediar.")
else:
    columnas_numericas = [
        "tiempo_gtfs_min",
        "google_eta_min",
        "status_code",
    ]
    for columna in columnas_numericas:
        df_observaciones[columna] = pd.to_numeric(
            df_observaciones[columna], errors="coerce"
        )

    df_validas = df_observaciones[
        df_observaciones["condition"].eq("ROUTE_EXISTS")
        & df_observaciones["status_code"].fillna(0).eq(0)
        & df_observaciones["google_eta_min"].notna()
        & df_observaciones["tiempo_gtfs_min"].notna()
    ].copy()

    df_validas["desviacion_eta_min"] = (
        df_validas["google_eta_min"] - df_validas["tiempo_gtfs_min"]
    )
    df_validas["atraso_estimado_min"] = (
        df_validas["desviacion_eta_min"].clip(lower=0)
    )

    paradas_consulta = set(df_consulta["stop_id"])
    paradas_historicas_cubiertas = set(df_validas["stop_id"]) & paradas_consulta
    cobertura = len(paradas_historicas_cubiertas) / max(len(paradas_consulta), 1)
    print(f"Observaciones totales: {len(df_observaciones):,}")
    print(f"Observaciones válidas: {len(df_validas):,}")
    print(f"Cobertura de paradas de esta población: {cobertura:.1%}")
    print(f"Datos de rutas: Google Maps ©{datetime.now().year} Google")
    display(
        df_validas[
            [
                "observed_at_utc",
                "stop_id",
                "route_id_principal",
                "tiempo_gtfs_min",
                "google_eta_min",
                "desviacion_eta_min",
                "atraso_estimado_min",
            ]
        ].head(10)
    )


Histórico alineado con la red GTFS actual: ../data/google_maps_transit_observaciones.csv
Observaciones totales: 7,698
Observaciones válidas: 7,698
Cobertura de paradas de esta población: 100.0%
Datos de rutas: Google Maps ©2026 Google


,observed_at_utc,stop_id,route_id_principal,tiempo_gtfs_min,google_eta_min,desviacion_eta_min,atraso_estimado_min
0,2026-09-14T16:47:07.157796+00:00,2982,350.0,25.850000,17.733333,-8.116667,0.00
1,2026-09-14T16:47:07.157796+00:00,2981,325.0,28.383333,14.500000,-13.883333,0.00
2,2026-09-14T16:47:07.157796+00:00,3297,97.0,26.825000,16.600000,-10.225000,0.00
3,2026-09-14T16:47:07.157796+00:00,3222,348.0,24.850000,14.016667,-10.833333,0.00
4,2026-09-14T16:47:07.157796+00:00,3294,344.0,3.583333,13.283333,9.700000,9.70
5,2026-09-14T16:47:07.157796+00:00,3284,324.0,12.650000,21.000000,8.350000,8.35
6,2026-09-14T16:47:07.157796+00:00,3571,336.0,24.833333,14.466667,-10.366667,0.00
7,2026-09-14T16:47:07.157796+00:00,1920,324.0,23.200000,19.500000,-3.700000,0.00
8,2026-09-14T16:47:07.157796+00:00,3295,344.0,2.216667,5.466667,3.250000,3.25
9,2026-09-14T16:47:07.157796+00:00,5196,324.0,22.683333,23.533333,0.850000,0.85


## 8. Promedio del atraso y tiempo corregido

El atraso proxy se agrega por `route_id_principal`. La corrección solicitada es:

```text
tiempo_corregido = tiempo_GTFS + atraso_promedio_de_la_ruta
```

También se calcula la ETA media observada por parada como control. Cuando una ruta aún no tiene observaciones válidas no se inventa un atraso: queda en cero y `correccion_disponible=False`.


In [9]:
PROMEDIO_RUTAS_PATH = DATA_DIR / "promedio_atrasos_google_por_ruta.csv"
CORRECCIONES_PATH = DATA_DIR / "tiempos_corregidos_google.csv"

if df_validas.empty:
    promedio_por_ruta = pd.DataFrame(
        columns=[
            "route_id_principal",
            "observaciones",
            "paradas_observadas",
            "atraso_promedio_ruta_min",
            "desviacion_promedio_ruta_min",
        ]
    )
    correcciones = df_base.copy()
    correcciones["atraso_promedio_ruta_min"] = 0.0
    correcciones["google_eta_promedio_stop_min"] = np.nan
    correcciones["observaciones_stop"] = 0
    correcciones["correccion_disponible"] = False
    correcciones["tiempo_corregido_min"] = correcciones["tiempo_gtfs_min"]
else:
    promedio_por_ruta = (
        df_validas.dropna(subset=["route_id_principal"])
        .groupby("route_id_principal", as_index=False)
        .agg(
            observaciones=("stop_id", "size"),
            paradas_observadas=("stop_id", "nunique"),
            atraso_promedio_ruta_min=("atraso_estimado_min", "mean"),
            desviacion_promedio_ruta_min=("desviacion_eta_min", "mean"),
        )
    )
    promedio_por_ruta["route_short_name"] = promedio_por_ruta[
        "route_id_principal"
    ].map(route_name_map)

    promedio_por_stop = (
        df_validas.groupby("stop_id", as_index=False)
        .agg(
            google_eta_promedio_stop_min=("google_eta_min", "mean"),
            observaciones_stop=("google_eta_min", "size"),
        )
    )

    correcciones = (
        df_base.merge(
            promedio_por_ruta[
                ["route_id_principal", "atraso_promedio_ruta_min"]
            ],
            on="route_id_principal",
            how="left",
        )
        .merge(promedio_por_stop, on="stop_id", how="left")
    )
    correcciones["correccion_disponible"] = correcciones[
        "atraso_promedio_ruta_min"
    ].notna()
    correcciones["atraso_promedio_ruta_min"] = correcciones[
        "atraso_promedio_ruta_min"
    ].fillna(0)
    correcciones["observaciones_stop"] = correcciones[
        "observaciones_stop"
    ].fillna(0).astype(int)
    correcciones["tiempo_corregido_min"] = (
        correcciones["tiempo_gtfs_min"]
        + correcciones["atraso_promedio_ruta_min"]
    )

    promedio_por_ruta.to_csv(PROMEDIO_RUTAS_PATH, index=False)
    correcciones.to_csv(CORRECCIONES_PATH, index=False)
    print(f"Promedios exportados: {PROMEDIO_RUTAS_PATH}")
    print(f"Correcciones exportadas: {CORRECCIONES_PATH}")

display(promedio_por_ruta.sort_values("atraso_promedio_ruta_min", ascending=False).head(15))


Promedios exportados: ../data/promedio_atrasos_google_por_ruta.csv
Correcciones exportadas: ../data/tiempos_corregidos_google.csv


,route_id_principal,observaciones,paradas_observadas,atraso_promedio_ruta_min,desviacion_promedio_ruta_min,route_short_name
85,408.0,28,4,13.071429,12.259226,C73
79,377.0,63,9,11.727778,11.727778,T09
76,374.0,99,14,10.938721,9.239731,T06
67,364.0,49,7,7.969048,7.969048,L159
9,128.0,7,1,7.785714,7.785714,L100
73,371.0,131,17,7.506743,6.124300,T03
19,245.0,49,7,7.157483,6.346939,C58
56,350.0,231,31,6.686291,3.662987,C62
21,256.0,79,11,6.364768,4.355485,T12
4,95.0,7,1,6.214286,5.916667,L56


## 9. Paradas y rutas dentro de 30 minutos

Este reporte usa `tiempo_corregido_min`, no el horario GTFS sin ajustar. Se excluyen del resultado final las filas sin corrección disponible para evitar afirmar que están dentro de 30 minutos sin evidencia de Google. Para diagnóstico se muestra además cuántas quedaron pendientes de observación.


In [10]:
rutas_30_min = correcciones[
    correcciones["correccion_disponible"]
    & correcciones["tiempo_corregido_min"].le(30)
].copy()

rutas_30_min = rutas_30_min.sort_values(
    ["tiempo_corregido_min", "route_short_name", "stop_name"]
)

pendientes_30_gtfs = correcciones[
    ~correcciones["correccion_disponible"]
    & correcciones["tiempo_gtfs_min"].le(30)
]

print(f"Paradas confirmadas a 30 min o menos: {len(rutas_30_min):,}")
print(
    "Paradas que GTFS coloca a 30 min o menos pero aún no tienen corrección:",
    f"{len(pendientes_30_gtfs):,}",
)

columnas_reporte = [
    "stop_id",
    "stop_name",
    "route_id_principal",
    "route_short_name",
    "tiempo_gtfs_min",
    "atraso_promedio_ruta_min",
    "tiempo_corregido_min",
    "google_eta_promedio_stop_min",
    "observaciones_stop",
]
display(rutas_30_min[columnas_reporte].reset_index(drop=True))


Paradas confirmadas a 30 min o menos: 113
Paradas que GTFS coloca a 30 min o menos pero aún no tienen corrección: 1


,stop_id,stop_name,route_id_principal,route_short_name,tiempo_gtfs_min,atraso_promedio_ruta_min,tiempo_corregido_min,google_eta_promedio_stop_min,observaciones_stop
0,2341,Av. Felipe Ángeles/Av. San Roque,344.0,C54,1.533333,4.932721,6.466054,6.454167,8
1,3295,Felipe Ángeles/Fraternidad,344.0,C54,2.216667,4.932721,7.149387,8.841667,8
2,2340,Av. Felipe Ángeles/Plan de Ayala Poniente,344.0,C54,2.733333,4.932721,7.666054,8.300000,8
3,3294,Av. Felipe Ángeles/Calle del Porvenir,344.0,C54,3.583333,4.932721,8.516054,10.381250,8
4,2339,Av. Felipe Ángeles/Felipe Ángeles 225,344.0,C54,4.200000,4.932721,9.132721,9.570833,8
...,...,...,...,...,...,...,...,...,...
108,3122,Calle Cerro del Peñón/Cerro del Tezcaltzin,341.0,C51,27.033333,2.783113,29.816447,26.529167,8
109,3084,Calle Torneros/Calle Obreros,345.0,C55,27.983333,1.844662,29.827996,33.125000,8
110,2882,Av. Cerro Sombrerete/Punta Esmeralda,371.0,T03,22.366667,7.506743,29.873410,34.477083,8
111,2331,Cetis 16,350.0,C62,23.233333,6.686291,29.919625,33.177083,8


## 10. Recomendación de operación

Para que el promedio represente horas pico y valle, ejecute este notebook de forma programada varias veces al día y conserve el histórico. Como mínimo conviene muestrear mañana, mediodía y tarde durante días hábiles y fines de semana.

Antes de automatizar a gran escala:

- configure una cuota diaria y alertas de presupuesto en Google Cloud;
- revise la cobertura de QroBus en Google Transit;
- no publique la llave ni el archivo de observaciones sin revisar las condiciones de uso de Google Maps;
- si se obtiene un feed GTFS-Realtime oficial, reemplace el proxy por el atraso de `TripUpdates`.
